<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Batch and streaming

**[Batch and streaming](https://learning.nextia-ai.com/courses/speech/m09/batch-and-streaming/)** · Speech AI: Transcribe, Understand and Speak · Module 9, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** turn recorded compute times into a real-time factor and a cost per audio hour, build a batch worker that can stop and start again without doing a call twice, and find the end of each caller turn while the audio streams in, as the IVR does.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. Whisper tiny is downloaded once (75 MB). |
| **You should know** | The real-time factor, from [Whisper sizes on a laptop](https://learning.nextia-ai.com/courses/speech/m04/whisper-sizes-on-a-laptop/), and the energy VAD, from [Speech or silence?](https://learning.nextia-ai.com/courses/speech/m03/speech-or-silence/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m09/batch-and-streaming/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M09-L01-batch-and-streaming/batch-and-streaming-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, faster-whisper 1.2.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. faster-whisper brings its own engine (CTranslate2) and the Silero voice detector: the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2 faster-whisper==1.2.1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile', 'faster-whisper']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 5 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "0ab1c11490f1e0341f11f9cf0fc061abde616a9297fbc830cde001ef60daa01a",
    "data/costs.json": "71133d6f2b813a316d00b7aa965e8dfe66037c7936ef6b2e298ebe2578f8174a",
    "data/recordings/asr/whisper-tiny.jsonl": "6c6026f6b4971dd04e945beeb3d4a155eacc6ba54d77452dff27f5082c9f3104",
    "data/recordings/asr/whisper-base.jsonl": "206b63ff863dd96fcbadf8034a8001e902ce06335cadf5348631b1a8d5d90420",
    "data/recordings/asr/whisper-small.jsonl": "67f27c0e6f5aa73d0e67cdb4b69c0e2d22c5f03a5dd74be14cbd8db0df17cb3f",
    "data/calls/phone/C-0245.wav": "ba5d7480e95efbd6f3b607e7aa2d1ffec2a00463c78320b12fe9cebd3fa5afc7",
    "data/calls/phone/C-0128.wav": "0b13d91f03204848d364c6ecf7adef4c05fc11ab68da9c226e543fa0cf95213a",
    "data/calls/phone/C-0225.wav": "0eab5a465ce8234b1c6e8354f7447cdaa5434eef6d7de6ccfb09ad0978d699b0",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Two jobs: after the call, and during the call

Larkfield records about 4,000 calls a month. Grace wants every transcript by the next morning: nobody waits for one call, so the calls can wait in a **queue** for a **worker**. That is the **batch** path. The out-of-hours phone line is different: the IVR must decide while the caller speaks, from small pieces of audio. That is the **streaming** path.

The batch path cares about cost: how much compute one hour of audio needs. The course recorded how long each Whisper size took on the 60 test calls. The **real-time factor** (RTF) is compute time divided by audio time. The **cost per audio hour** is RTF times the price of the server per hour, for a server that is busy all the time.

> **Predict.** Whisper small is about 2.5 times slower than Whisper base. What does that do to the cost per audio hour?

In [ ]:
calls = load_calls()
costs = json.load(open("data/costs.json"))
price = costs["cpu_server_usd_per_hour"]          # an assumption: a 4 vCPU server, on demand

def recorded_rtf(model, split="test", kind="phone"):
    """Real-time factor of a recorded run: all compute seconds / all audio seconds of the split."""
    rows = [json.loads(line) for line in open(f"data/recordings/asr/whisper-{model}.jsonl")]
    rows = [r for r in rows if r["kind"] == kind and calls[r["call_id"]]["split"] == split]
    compute = sum(r["seconds"] for r in rows)
    audio = sum(calls[r["call_id"]]["duration"] for r in rows)
    return compute / audio, len(rows), audio

rtfs = {}
for model in ["tiny", "base", "small"]:
    rtf, n, audio = recorded_rtf(model)
    rtfs[model] = rtf
    print(f"whisper-{model:6} {n} calls, {audio / 60:.0f} min of audio: RTF {rtf:.3f}, "
          f"cost per audio hour US${rtf * price:.4f}")

> **Check.** Whisper base: RTF 0.03, US$0.0051 an audio hour; Whisper small: RTF 0.075, US$0.0127. The cost grows exactly as the RTF does. The price of US$0.17 an hour is an assumption of the course, not a measurement.

## 2. Your turn: Larkfield's month

> **Your turn.** Write `monthly_cost(calls_per_month, minutes_per_call, rtf, usd_per_hour)`. It returns the US$ a month: the audio hours (calls × minutes ÷ 60) × the real-time factor × the price per hour. The lesson assumes 4,000 calls of 6 minutes: an upper bound for the audio, not a measurement.

In [ ]:
def monthly_cost(calls_per_month, minutes_per_call, rtf, usd_per_hour):
    ...  # your code
    return None

In [ ]:
month_base = monthly_cost(4000, 6, rtfs["base"], price)
month_small = monthly_cost(4000, 6, rtfs["small"], price)
if month_base is None:
    print("Write monthly_cost in the cell above first.")
else:
    print(f"Whisper base: US${month_base:.2f} a month; Whisper small: US${month_small:.2f} a month")

In [ ]:
expect('month_base', month_base, 2.056121147321452, tolerance=0.001)

> **Check.** About US$2.06 a month with Whisper base and US$5.07 with Whisper small. At this size, local compute costs a few dollars; the lesson compares it with hosted prices.

## 3. A queue that survives a crash

A batch worker runs for hours. If it stops in the middle, it must start again where it stopped: no call twice, no call lost. The trick: write one line to the results file **after each call**, and skip every call that is already in the file.

The next cell downloads Whisper tiny once and copies three test calls into a folder `inbox`.

In [ ]:
import time, shutil
from pathlib import Path
from faster_whisper import WhisperModel
from huggingface_hub import snapshot_download

model = WhisperModel(snapshot_download("Systran/faster-whisper-tiny", revision="d90ca5fe260221311c53c58e660288d3deb8d356"),
                     device="cpu", compute_type="int8", cpu_threads=4)
inbox, results = Path("inbox"), Path("batch-results.jsonl")
shutil.rmtree(inbox, ignore_errors=True); inbox.mkdir()
results.unlink(missing_ok=True)
for cid in ['C-0245', 'C-0128', 'C-0225']:
    shutil.copy(f"data/calls/phone/{cid}.wav", inbox / f"{cid}.wav")
print(sorted(p.name for p in inbox.iterdir()))

The worker below stops after two calls in its first run, as if the computer crashed. Then it runs twice more.

> **Predict.** How many calls does the second run transcribe? And the third?

In [ ]:
def run_worker(inbox, results, stop_after=None):
    """Transcribe every call in the inbox that is not in the results file yet; one line per call, written at once.
    stop_after: stop (as if the computer crashed) after this many new calls."""
    done = set()
    if results.exists():
        done = {json.loads(line)["file"] for line in results.read_text().splitlines() if line}
    stats = {"calls": 0, "skipped": 0, "audio": 0.0, "compute": 0.0}
    with results.open("a", encoding="utf-8") as f:
        for p in sorted(inbox.glob("*.wav")):
            if p.name in done:
                stats["skipped"] += 1
                continue
            if stop_after is not None and stats["calls"] == stop_after:
                print("...the worker stops here (a crash)")
                break
            x, sr = read_audio(p, 16000)
            t0 = time.perf_counter()
            segments, _ = model.transcribe(x, language="en", beam_size=1, condition_on_previous_text=False)
            text = " ".join(s.text.strip() for s in segments)
            dt = time.perf_counter() - t0
            f.write(json.dumps({"file": p.name, "text": text, "seconds": round(dt, 2)}) + "\n")
            f.flush()                                   # the line is on disk before the next call starts
            stats["calls"] += 1; stats["audio"] += len(x) / sr; stats["compute"] += dt
    return stats

first = run_worker(inbox, results, stop_after=2)
print("first run: ", first["calls"], "new,", first["skipped"], "skipped")
second = run_worker(inbox, results)
print("second run:", second["calls"], "new,", second["skipped"], "skipped")
third = run_worker(inbox, results)
print("third run: ", third["calls"], "new,", third["skipped"], "skipped")
lines = results.read_text().splitlines()
print(len(lines), "lines in the results file")

> **Check.** The first run did 2 calls. The second run did 1 new call and skipped 2. The third run skipped all 3. The results file has 3 lines: one per call, none twice. A worker that is **safe to run again** like this is called *idempotent*.

## 4. Failure case: results written at the end

A common mistake is to keep the results in a list and write the file once, at the end. Then a crash at call 300 of 500 loses all 300 transcripts, and the next run starts from zero. The `f.flush()` in the worker matters too: it pushes each line to the disk at once.

## 5. Streaming: decide while the caller speaks

The phone line delivers audio in small pieces. The next cell reads the caller's side of call C-0225 in pieces of 200 ms. After each piece, it looks at the new 30 ms frames and asks: speech or not? A turn ends when the caller has been quiet for 0.8 seconds. That is the IVR's setting.

In [ ]:
FRAME = 0.03  # 30 ms frames

def frame_db(x, sr):
    """The loudness of each 30 ms frame, in dBFS."""
    n = int(sr * FRAME)
    frames = x[: len(x) // n * n].reshape(-1, n)
    return 20 * np.log10(np.sqrt(np.mean(frames ** 2, axis=1)) + 1e-9)

def regions(flags, hangover=0.3, min_speech=0.1):
    """Speech frames -> (start, end) times. A pause shorter than `hangover` seconds stays inside the speech."""
    out, start, quiet = [], None, 0
    keep = int(round(hangover / FRAME))
    for i, f in enumerate(flags):
        if f:
            start = i if start is None else start
            quiet = 0
        elif start is not None:
            quiet += 1
            if quiet > keep:
                end = i - quiet + 1
                if (end - start) * FRAME >= min_speech:
                    out.append((start * FRAME, end * FRAME))
                start, quiet = None, 0
    if start is not None:
        out.append((start * FRAME, (len(flags) - quiet) * FRAME))
    return out

def energy_vad(x, sr, threshold_db=-40.0, hangover=0.3):
    """The baseline: a frame is speech when it is louder than threshold_db."""
    return regions(frame_db(x, sr) > threshold_db, hangover)

In [ ]:
call = calls["C-0225"]
caller, sr = read_audio("data/calls/phone/C-0225.wav", channel=1)    # the caller's side of a stereo call, 8 kHz
PIECE = 0.2                                                          # the audio arrives in 200 ms pieces

def stream_turn_ends(x, sr, end_of_turn=0.8, threshold_db=-40.0):
    """Read the audio piece by piece, as a phone line delivers it. After each piece, look at the new 30 ms frames:
    speech or not (the energy VAD of Module 3). A turn ends after `end_of_turn` seconds without speech."""
    ends, speaking, quiet, heard = [], False, 0.0, 0
    n = int(sr * FRAME)
    for k in range(int(len(x) / (PIECE * sr))):
        buffer = x[: int((k + 1) * PIECE * sr)]                       # everything received so far
        loud = frame_db(buffer, sr)[heard:] > threshold_db             # only the frames that are new
        heard += len(loud)
        for is_speech in loud:
            if is_speech:
                speaking, quiet = True, 0.0
            elif speaking:
                quiet += FRAME
                if quiet >= end_of_turn:
                    ends.append(round((k + 1) * PIECE, 1))           # decided at the end of this piece
                    speaking, quiet = False, 0.0
    return ends

ends = stream_turn_ends(caller, sr)
truth = [round(t["words"][-1]["end"], 2) for t in call["turns"] if t["speaker"] == "caller"]
print("caller turns end (labels):  ", truth)
print("turn end decided (streaming):", ends)

> **Check.** The labels say that the caller's 2 turns end at [9.44, 19.95] seconds. The streaming check decided [10.2, 20.8]: each one about 0.8 seconds later, because it waits 0.8 seconds of silence and decides only at the end of a 200 ms piece. That wait is part of every answer the IVR gives.

## 6. Change one thing: a shorter wait

> **Predict.** With an end of turn of 0.3 s instead of 0.8 s, how much sooner does the check decide that a turn has ended?

In [ ]:
ends_fast = stream_turn_ends(caller, sr, end_of_turn=0.3)
print("end of turn 0.8 s:", ends)
print("end of turn 0.3 s:", ends_fast)

> **Check.** With 0.3 s, the check decides [9.8, 20.2] instead of [10.2, 20.8]: about half a second sooner for each turn. On this call, no pause inside a caller's sentence is long enough to cut a turn in two. The course's IVR replay measured the setting on 60 calls: with an end of turn of 0.5 s, 38 calls passed, against 39 with 0.8 s. A shorter wait answers sooner, and risks answering in the middle of a sentence. [Turn-taking and interruptions](https://learning.nextia-ai.com/courses/speech/m08/turn-taking-and-interruptions/) explains this trade-off.

## Next

You computed the cost of the batch path, built a worker that is safe to run again, and found turn ends in a stream. The [lesson page](https://learning.nextia-ai.com/courses/speech/m09/batch-and-streaming/) compares the costs with a GPU and hosted services, and has the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Personal data in voice](https://learning.nextia-ai.com/courses/speech/m09/personal-data-in-voice/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m09/batch-and-streaming/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.